In [10]:
from pathlib import Path
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

In [11]:
base_dir_path = Path(
    "/eodc/private/tuwgeo/users/mabdelaa/repos/vv_vh_fusion/data/EQUI7_EU020M"
)

imgs = [
    img
    for img in base_dir_path.glob("*/*.tif")
    if "FLOOD-EXPF" in img.name and "VV" in img.name
]

for img in tqdm(imgs):
    print(img.name)

  0%|          | 0/8 [00:00<?, ?it/s]

FLOOD-EXPF_20181021T051751__VV_D095_E048N012T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20230609T153657__VV_A014_E066N012T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20230609T153607__VV_A014_E066N012T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20230609T153632__VV_A014_E066N012T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20210719T051848__VV_D095_E048N015T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20230609T153607__VV_A014_E063N012T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20230609T153657__VV_A014_E063N012T3_EU020M_V1M3R1_S1_TUWIEN.tif
FLOOD-EXPF_20230609T153632__VV_A014_E063N012T3_EU020M_V1M3R1_S1_TUWIEN.tif


In [12]:
def get_corresponding_file_paths(flood_vv):
    flood_vh = flood_vv.parent / flood_vv.name.replace("VV", "VH")
    uncert_vv = flood_vv.parent / flood_vv.name.replace("FLOOD", "UNCERT")
    uncert_vh = flood_vv.parent / flood_vv.name.replace("FLOOD", "UNCERT").replace(
        "VV", "VH"
    )
    flood_likelihood_vv = flood_vv.parent / flood_vv.name.replace(
        "FLOOD", "FLOOD_LIKELIHOOD"
    )
    flood_likelihood_vh = flood_vv.parent / flood_vv.name.replace(
        "FLOOD", "FLOOD_LIKELIHOOD"
    ).replace("VV", "VH")
    flood_probability_vv = flood_vv.parent / flood_vv.name.replace(
        "FLOOD", "FLOOD_PROBABILITY"
    )
    flood_probability_vh = flood_vv.parent / flood_vv.name.replace(
        "FLOOD", "FLOOD_PROBABILITY"
    ).replace("VV", "VH")

    # chech if files exist
    for file_path in [
        flood_vv,
        flood_vh,
        uncert_vv,
        uncert_vh,
        flood_likelihood_vv,
        flood_likelihood_vh,
        flood_probability_vv,
        flood_probability_vh,
    ]:
        assert file_path.exists(), f"File {file_path} does not exist."

    return {
        "flood_vv": flood_vv,
        "flood_vh": flood_vh,
        "uncert_vv": uncert_vv,
        "uncert_vh": uncert_vh,
        "flood_likelihood_vv": flood_likelihood_vv,
        "flood_likelihood_vh": flood_likelihood_vh,
        "flood_probability_vv": flood_probability_vv,
        "flood_probability_vh": flood_probability_vh,
    }


def read_rasters(paths):
    arrays = {}

    # Read metadata from flood_vv only
    with rasterio.open(paths["flood_vv"]) as src:
        arrays["flood_vv"] = src.read(1)
        meta = src.meta

    for key, path in paths.items():
        if key == "flood_vv":
            continue
        with rasterio.open(path) as src:
            arrays[key] = src.read(1)

    return arrays, meta


def fuse_flood_category(
    flood_vv_data, flood_vh_data, flood_likelihood_vv_data, flood_likelihood_vh_data
):

    vv_mask = flood_vv_data == 1
    vh_only_mask = (flood_vh_data == 1) & (~vv_mask)

    fused_flood = np.zeros_like(flood_vv_data, dtype=np.uint8)
    fused_flood[vv_mask] = 1
    fused_flood[vh_only_mask] = 2

    fused_likelihood = np.zeros_like(
        flood_likelihood_vv_data, dtype=flood_likelihood_vv_data.dtype
    )
    fused_likelihood[vv_mask] = flood_likelihood_vv_data[vv_mask]
    fused_likelihood[vh_only_mask] = flood_likelihood_vh_data[vh_only_mask]
    return fused_flood, fused_likelihood

In [13]:
for img_path in tqdm(imgs):
    paths = get_corresponding_file_paths(img_path)
    arrays, meta = read_rasters(paths)
    fused_flood, fused_likelihood = fuse_flood_category(
        arrays["flood_vv"],
        arrays["flood_vh"],
        arrays["flood_likelihood_vv"],
        arrays["flood_likelihood_vh"],
    )
    save_path = base_dir_path / "Fusion/Catagory_Based"
    save_path.mkdir(parents=True, exist_ok=True)
    fused_flood_path = save_path / img_path.name.replace(
        "FLOOD-EXPF", "FUSED-FLOOD-EXPF"
    ).replace("VV", "FUSED")
    fused_likelihood_path = save_path / img_path.name.replace(
        "FLOOD-EXPF", "FUSED-FLOOD-LIKELIHOOD"
    ).replace("VV", "FUSED")
    with rasterio.open(fused_flood_path, "w", **meta) as dst:
        dst.write(fused_flood, 1)
    with rasterio.open(fused_likelihood_path, "w", **meta) as dst:
        dst.write(fused_likelihood, 1)

  0%|          | 0/8 [00:00<?, ?it/s]